# Z_2_3_Formatting_Industry_Return_17_Cat 

**Objective:**
This notebook processes the Fama-French 17-industry portfolio dataset by extracting, reshaping, and enriching multiple financial time series sections. The goal is to transform raw, multi-section CSV data into structured, analysis-ready formats for integration with the SEO underpricing model.

**Key tasks include:**

- Parsing and extracting individual sections (e.g., value-weighted/ equal-weighted returns, BE/ME ratios)

- Timestamping period identifiers (e.g., YYYYMM → end-of-month datetime)

- Reshaping datasets into long format for industry-level analysis

- Merging with industry classification mappings

In [1]:
# ===========================================
# 1. Import Required Libraries
# ===========================================
import pandas as pd
from io import StringIO
import os


In [2]:
# ===========================================
# 2. Check for Malformed Lines in Raw CSV
# ===========================================

file_path = r"data\17_Industry_Portfolios.CSV"
min_expected_columns = 5

# Scan the file line-by-line to detect issues
with open(file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        columns = line.strip().split(",")
        if len(columns) < min_expected_columns:
            print(f"⚠️ Bad line at index {i} (only {len(columns)} columns):")
            print(repr(line.strip()))


FileNotFoundError: [Errno 2] No such file or directory: 'data\\17_Industry_Portfolios.CSV'

In [39]:
# ===========================================
# 3. Split Raw File into Multiple Sections and Save as Parquet
# ===========================================

# Define the output directory for cleaned parquet files
output_dir = r"E:\...path...\Portfolio 17\Formatted"
os.makedirs(output_dir, exist_ok=True)

# Read full file into lines
with open(file_path, "r") as f:
    lines = f.readlines()

# Define headers and output filenames
section_headers = {
    "Average Value Weighted Returns -- Monthly": "value_weighted_monthly.parquet",
    "Average Equal Weighted Returns -- Monthly": "equal_weighted_monthly.parquet",
    "Average Value Weighted Returns -- Annual": "value_weighted_annual.parquet",
    "Average Equal Weighted Returns -- Annual": "equal_weighted_annual.parquet",
    "Number of Firms in Portfolios": "num_firms.parquet",
    "Average Firm Size": "avg_firm_size.parquet",
    "Sum of BE / Sum of ME": "sum_be_me.parquet",
    "Value-Weighted Average of BE/ME": "vw_avg_be_me.parquet"
}

# Locate the start of each section
section_starts = {}
for i, line in enumerate(lines):
    if line.strip() in section_headers:
        section_starts[line.strip()] = i

# Determine line ranges for each section
sorted_sections = sorted(section_starts.items(), key=lambda x: x[1])
sections = []
for i in range(len(sorted_sections)):
    header, start_line = sorted_sections[i]
    end_line = sorted_sections[i + 1][1] if i + 1 < len(sorted_sections) else len(lines)
    sections.append((header, start_line + 1, end_line))  # +1 to skip title

# Loop through sections and save each as a Parquet file
for section_name, start, end in sections:
    block = lines[start:end]
    block_cleaned = [line for line in block if line.strip()]
    block_str = StringIO("".join(block_cleaned))

    try:
        df = pd.read_csv(block_str)
        output_filename = os.path.join(output_dir, section_headers[section_name])
        df.to_parquet(output_filename, index=False)
        print(f"✅ Exported '{section_name}' to {output_filename}")
    except Exception as e:
        print(f"❌ Failed to parse section '{section_name}': {e}")


✅ Exported 'Average Value Weighted Returns -- Monthly' to E:\BUSINFO 718 - Finacle_F2 - SEO Underpricing\Python Environment\New folder\BUSINFO 718 - Finacle_F2 - SEO Underpricing\Python Environment\data\Portfolio 17\Formatted\value_weighted_monthly.parquet
✅ Exported 'Average Equal Weighted Returns -- Monthly' to E:\BUSINFO 718 - Finacle_F2 - SEO Underpricing\Python Environment\New folder\BUSINFO 718 - Finacle_F2 - SEO Underpricing\Python Environment\data\Portfolio 17\Formatted\equal_weighted_monthly.parquet
✅ Exported 'Average Value Weighted Returns -- Annual' to E:\BUSINFO 718 - Finacle_F2 - SEO Underpricing\Python Environment\New folder\BUSINFO 718 - Finacle_F2 - SEO Underpricing\Python Environment\data\Portfolio 17\Formatted\value_weighted_annual.parquet
✅ Exported 'Average Equal Weighted Returns -- Annual' to E:\BUSINFO 718 - Finacle_F2 - SEO Underpricing\Python Environment\New folder\BUSINFO 718 - Finacle_F2 - SEO Underpricing\Python Environment\data\Portfolio 17\Formatted\equal_

In [ ]:
# ===========================================
# 4. Process Equal Weighted Monthly Data
# ===========================================

# Load parquet file with monthly returns
equal_weighted_monthly = pd.read_parquet(os.path.join(output_dir, 'value_weighted_monthly.parquet'))

# Convert period string (e.g., 192607) to datetime (end-of-month)
equal_weighted_monthly['Date'] = pd.to_datetime(equal_weighted_monthly['Unnamed: 0'].astype(str), format='%Y%m') + pd.offsets.MonthEnd(0)

# Drop the old column and reorder for clarity
equal_weighted_monthly.drop(columns='Unnamed: 0', inplace=True)
cols = ['Date'] + [col for col in equal_weighted_monthly.columns if col != 'Date']
equal_weighted_monthly = equal_weighted_monthly[cols]

equal_weighted_monthly.info()

# Save updated dataset
output_file_path = os.path.join(output_dir, 'equal_weighted_monthly.parquet')
equal_weighted_monthly.to_parquet(output_file_path)
print(f"✅ File saved successfully to {output_file_path}")


In [41]:
# ===========================================
# 5. Load Industry Mapping Excel Sheet
# ===========================================

# Load second sheet (sheet index = 1)
portfolio_mappings = pd.read_excel(
    r'E:\...path...\Industries_Cat17 mappings.xlsx', sheet_name=1
)

# Rename columns for clarity
portfolio_mappings.rename(columns={'Industries': 'Industry', '17 cat': 'Portfolio'}, inplace=True)


,Date,Food,Mines,Oil,Clths,Durbl,Chems,Cnsum,Cnstr,Steel,FabPr,Machn,Cars,Trans,Utils,Rtail,Finan,Other
0,1926-07-31,0.48,3.78,-1.41,6.02,-1.62,8.46,1.42,2.31,4.07,8.77,3.79,17.43,1.83,7.04,0.13,-0.02,1.22
1,1926-08-31,2.91,0.69,3.60,0.15,-1.96,5.70,5.84,4.33,2.17,-5.56,2.35,3.96,4.54,-1.69,-0.68,4.47,3.11
2,1926-09-30,1.20,1.10,-3.68,0.26,0.24,5.48,1.21,-0.06,0.15,-4.13,-0.65,5.57,0.32,2.04,0.21,-1.61,1.82
3,1926-10-31,-3.06,-0.79,-1.02,0.37,-6.07,-4.76,0.69,-4.79,-3.85,-5.13,-3.29,-8.29,-2.90,-2.63,-2.26,-5.51,-0.88
4,1926-11-30,6.37,4.38,-0.01,2.22,-1.95,5.27,4.63,2.45,3.86,3.57,4.54,-0.28,2.19,3.71,6.44,2.34,1.38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1177,2024-08-31,3.78,-3.11,-2.92,5.31,-0.97,0.46,8.05,0.66,-6.85,4.90,0.19,-4.44,2.31,4.55,1.71,2.50,2.59
1178,2024-09-30,1.11,7.33,-3.36,5.52,2.13,3.99,-3.32,6.89,1.84,5.27,2.84,12.27,1.30,5.35,3.72,-0.51,2.63
1179,2024-10-31,-5.04,-3.35,-0.46,-6.72,-2.45,-3.05,-3.63,-3.91,-1.52,-0.70,0.64,-3.26,-0.39,0.71,-0.91,2.64,-0.82
1180,2024-11-30,0.71,-0.70,6.51,7.99,4.42,0.56,0.94,8.87,10.48,18.60,3.27,28.31,6.43,6.18,10.04,12.03,6.16


In [ ]:
# ===========================================
# 6. Reshape and Merge Portfolio Returns with Mapping
# ===========================================

# Melt from wide to long format
vw_monthly = pd.read_parquet(os.path.join(output_dir, 'equal_weighted_monthly.parquet'))
vw_monthly = vw_monthly.melt(id_vars=['Date'], var_name='Portfolio', value_name='Values')

# Merge portfolio returns with industry labels
portfolio_merged = vw_monthly.merge(portfolio_mappings, on='Portfolio', how='left')

# Save final merged portfolio data
output_file_path = os.path.join(output_dir, 'Portfolio_Mapping_Merged.parquet')
portfolio_merged.to_parquet(output_file_path)
print(f"✅ File saved successfully to {output_file_path}")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1182 entries, 0 to 1181
Data columns (total 18 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    1182 non-null   datetime64[ns]
 1   Food    1182 non-null   float64       
 2   Mines   1182 non-null   float64       
 3   Oil     1182 non-null   float64       
 4   Clths   1182 non-null   float64       
 5   Durbl   1182 non-null   float64       
 6   Chems   1182 non-null   float64       
 7   Cnsum   1182 non-null   float64       
 8   Cnstr   1182 non-null   float64       
 9   Steel   1182 non-null   float64       
 10  FabPr   1182 non-null   float64       
 11  Machn   1182 non-null   float64       
 12  Cars    1182 non-null   float64       
 13  Trans   1182 non-null   float64       
 14  Utils   1182 non-null   float64       
 15  Rtail   1182 non-null   float64       
 16  Finan   1182 non-null   float64       
 17  Other   1182 non-null   float64       
dtypes: datet